<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/AOFE_V13_5_ControlledScaling_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# CELL 01 — IMPORTS + CONFIG
# ============================================

import json
import math
import numpy as np
from collections import deque

DATA_PATH = "/content/arc-agi_training_challenges.json"

print("CELL 01 loaded.")

CELL 01 loaded.


In [2]:
# ============================================
# CELL 02 — LOAD DATASET + CONVERT FORMAT
# ============================================

def convert_to_solver_format(tasks):
    solver_tasks = {}

    for task_id, task_data in tasks.items():
        if isinstance(task_data, dict) and "train" in task_data and "test" in task_data:
            solver_tasks[task_id] = {
                "train": [
                    {
                        "input": [row[:] for row in pair["input"]],
                        "output": [row[:] for row in pair["output"]],
                    }
                    for pair in task_data["train"]
                ],
                "test": [
                    {
                        "input": [row[:] for row in pair["input"]],
                    }
                    for pair in task_data["test"]
                ],
            }
        elif isinstance(task_data, list):
            solver_tasks[task_id] = {
                "train": [
                    {
                        "input": [row[:] for row in pair["input"]],
                        "output": [row[:] for row in pair["output"]],
                    }
                    for pair in task_data
                    if "input" in pair and "output" in pair
                ],
                "test": [],
            }
        else:
            raise ValueError(f"Unsupported task format for {task_id}")

    return solver_tasks


with open(DATA_PATH, "r") as f:
    raw_tasks = json.load(f)

solver_tasks = convert_to_solver_format(raw_tasks)

sample_id = list(solver_tasks.keys())[0]
sample_task = solver_tasks[sample_id]

print("Loaded raw tasks:", len(raw_tasks))
print("Converted tasks:", len(solver_tasks))
print("Sample task id:", sample_id)
print("Sample keys:", sample_task.keys())
print("Train pairs:", len(sample_task["train"]))
print("Test pairs:", len(sample_task["test"]))
print("CELL 02 loaded.")

Loaded raw tasks: 1000
Converted tasks: 1000
Sample task id: 00576224
Sample keys: dict_keys(['train', 'test'])
Train pairs: 2
Test pairs: 1
CELL 02 loaded.


In [3]:
# ============================================
# CELL 03 — GRID HELPERS
# ============================================

def copy_grid(grid):
    return [row[:] for row in grid]

def to_np(grid):
    return np.array(grid, dtype=int)

def to_list(grid):
    if isinstance(grid, np.ndarray):
        return grid.astype(int).tolist()
    return [list(map(int, row)) for row in grid]

def grid_shape(grid):
    return (len(grid), len(grid[0]) if grid else 0)

def same_shape(a, b):
    return len(a) == len(b) and len(a[0]) == len(b[0])

def unique_colors(grid):
    return sorted({v for row in grid for v in row})

print("CELL 03 loaded.")

CELL 03 loaded.


In [4]:
# ============================================
# CELL 04 — OBJECT EXTRACTION
# ============================================

def get_neighbors(r, c, h, w, connectivity=4):
    nbrs = [(r-1,c), (r+1,c), (r,c-1), (r,c+1)]
    if connectivity == 8:
        nbrs += [(r-1,c-1), (r-1,c+1), (r+1,c-1), (r+1,c+1)]
    return [(nr, nc) for nr, nc in nbrs if 0 <= nr < h and 0 <= nc < w]

def extract_objects(grid, background=0, connectivity=4):
    g = to_np(grid)
    h, w = g.shape
    visited = np.zeros((h, w), dtype=bool)
    objects = []

    for r in range(h):
        for c in range(w):
            if visited[r, c] or g[r, c] == background:
                continue

            color = int(g[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []

            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))

                for nr, nc in get_neighbors(cr, cc, h, w, connectivity):
                    if not visited[nr, nc] and g[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))

            rows = [p[0] for p in pixels]
            cols = [p[1] for p in pixels]

            obj = {
                "id": len(objects),
                "color": color,
                "pixels": pixels,
                "area": len(pixels),
                "bbox": (min(rows), min(cols), max(rows), max(cols)),
                "centroid": (sum(rows) / len(rows), sum(cols) / len(cols)),
                "touches_border": any(
                    pr == 0 or pc == 0 or pr == h - 1 or pc == w - 1
                    for pr, pc in pixels
                ),
            }
            objects.append(obj)

    return objects

print("CELL 04 loaded.")

CELL 04 loaded.


In [5]:
# ============================================
# CELL 05 — ROLE INFERENCE
# ============================================

def infer_object_roles_v2(objects, grid):
    if not objects:
        return {
            "largest": None,
            "smallest": None,
            "topmost": None,
            "bottommost": None,
            "leftmost": None,
            "rightmost": None,
            "center_most": None,
            "border_objects": [],
        }

    h, w = grid_shape(grid)
    center = ((h - 1) / 2.0, (w - 1) / 2.0)

    def center_dist(obj):
        cr, cc = obj["centroid"]
        return (cr - center[0]) ** 2 + (cc - center[1]) ** 2

    largest = max(objects, key=lambda o: (o["area"], -o["id"]))
    smallest = min(objects, key=lambda o: (o["area"], o["id"]))
    topmost = min(objects, key=lambda o: (o["bbox"][0], o["bbox"][1], o["id"]))
    bottommost = max(objects, key=lambda o: (o["bbox"][2], -o["bbox"][1], -o["id"]))
    leftmost = min(objects, key=lambda o: (o["bbox"][1], o["bbox"][0], o["id"]))
    rightmost = max(objects, key=lambda o: (o["bbox"][3], -o["bbox"][0], -o["id"]))
    center_most = min(objects, key=lambda o: (center_dist(o), o["id"]))
    border_objects = [o for o in objects if o["touches_border"]]

    return {
        "largest": largest,
        "smallest": smallest,
        "topmost": topmost,
        "bottommost": bottommost,
        "leftmost": leftmost,
        "rightmost": rightmost,
        "center_most": center_most,
        "border_objects": border_objects,
    }

print("CELL 05 loaded.")

CELL 05 loaded.


In [6]:
# ============================================
# CELL 06 — CORE OPERATORS
# ============================================

def rotate90(grid):
    return np.rot90(to_np(grid), 1)

def rotate180(grid):
    return np.rot90(to_np(grid), 2)

def rotate270(grid):
    return np.rot90(to_np(grid), 3)

def reflect_h(grid):
    return np.flipud(to_np(grid))

def reflect_v(grid):
    return np.fliplr(to_np(grid))

def color_map(grid, mapping):
    g = to_np(grid)
    out = g.copy()
    for src, dst in mapping.items():
        out[g == src] = dst
    return out

def conditional_color_map(grid, source, target):
    g = to_np(grid)
    out = g.copy()
    out[g == source] = target
    return out

def tile_full(grid, nx, ny):
    return np.tile(to_np(grid), (nx, ny))

def crop_to_bbox(grid):
    g = to_np(grid)
    rows = np.any(g != 0, axis=1)
    cols = np.any(g != 0, axis=0)

    if not np.any(rows) or not np.any(cols):
        return g.copy()

    r_idx = np.where(rows)[0]
    c_idx = np.where(cols)[0]
    return g[r_idx[0]:r_idx[-1]+1, c_idx[0]:c_idx[-1]+1]

def extract_largest_object(grid):
    g = to_np(grid)
    objs = extract_objects(g.tolist())
    if not objs:
        return g.copy()

    largest = max(objs, key=lambda o: (o["area"], -o["id"]))
    out = np.zeros_like(g)
    for r, c in largest["pixels"]:
        out[r, c] = g[r, c]
    return out

def recolor_role(grid, role, color):
    g = to_np(grid)
    objs = extract_objects(g.tolist())
    roles = infer_object_roles_v2(objs, g.tolist())

    if role not in roles or roles[role] is None:
        return g.copy()

    out = g.copy()
    for r, c in roles[role]["pixels"]:
        out[r, c] = color
    return out

def translate_role(grid, role, dx, dy, background=0):
    g = to_np(grid)
    objs = extract_objects(g.tolist(), background=background)
    roles = infer_object_roles_v2(objs, g.tolist())

    if role not in roles or roles[role] is None:
        return g.copy()

    obj = roles[role]
    out = g.copy()

    for r, c in obj["pixels"]:
        out[r, c] = background

    for r, c in obj["pixels"]:
        nr = r + dy
        nc = c + dx
        if 0 <= nr < g.shape[0] and 0 <= nc < g.shape[1]:
            out[nr, nc] = obj["color"]

    return out

print("CELL 06 loaded.")

CELL 06 loaded.


In [7]:
# ============================================
# CELL 07 — OPERATOR REGISTRY (UNIFIED + TASK-FILTERED)
# ============================================

def detect_task_features(task):
    pair = task["train"][0]
    inp = pair["input"]
    out = pair["output"]

    in_h, in_w = len(inp), len(inp[0])
    out_h, out_w = len(out), len(out[0])

    same_shape_flag = (in_h == out_h and in_w == out_w)
    expansion_flag = (out_h > in_h or out_w > in_w)

    color_change_flag = False
    overlap_h = min(in_h, out_h)
    overlap_w = min(in_w, out_w)

    for r in range(overlap_h):
        for c in range(overlap_w):
            if inp[r][c] != out[r][c]:
                color_change_flag = True
                break
        if color_change_flag:
            break

    return {
        "same_shape": same_shape_flag,
        "expansion": expansion_flag,
        "color_change": color_change_flag,
    }


def get_all_ops(task):
    out_colors = sorted({
        v
        for pair in task["train"]
        for row in pair["output"]
        for v in row
    })

    roles = [
        "largest",
        "smallest",
        "center_most",
        "topmost",
        "bottommost",
        "leftmost",
        "rightmost",
    ]

    ops = []

    # ----------------------------------
    # GEOMETRIC / STRUCTURAL
    # ----------------------------------
    ops.extend([
        ("rotate90", {}),
        ("rotate180", {}),
        ("rotate270", {}),
        ("reflect_h", {}),
        ("reflect_v", {}),
        ("tile_full", {"nx": 2, "ny": 2}),
        ("tile_full", {"nx": 3, "ny": 3}),
        ("crop_to_bbox", {}),
        ("extract_largest_object", {}),
        ("crop_nonzero_bbox", {}),
        ("isolate_objects", {}),
        ("compress_vertical", {}),
        ("compress_horizontal", {}),
    ])

    # ----------------------------------
    # COLOR OPS
    # ----------------------------------
    for s in range(10):
        for t in range(10):
            if s != t:
                ops.append(("conditional_color_map", {"source": s, "target": t}))

    # inferred global color map from first train pair
    pair0 = task["train"][0]
    inferred_map = {}
    consistent = True

    inp = pair0["input"]
    out = pair0["output"]

    if same_shape(inp, out):
        for r in range(len(inp)):
            for c in range(len(inp[0])):
                a = inp[r][c]
                b = out[r][c]

                if a in inferred_map and inferred_map[a] != b:
                    consistent = False
                    break
                inferred_map[a] = b

            if not consistent:
                break

    if consistent and inferred_map:
        ops.append(("color_map", {"mapping": inferred_map}))

    # ----------------------------------
    # ROLE OPS
    # ----------------------------------
    for role in roles:
        for color in out_colors[:8]:
            ops.append(("recolor_role", {"role": role, "color": color}))

        ops.append(("translate_role", {"role": role, "dx": 1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": -1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": 1}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": -1}))

    return ops


def build_task_ops(task):
    all_ops = get_all_ops(task)
    features = detect_task_features(task)

    filtered = []

    for op in all_ops:
        name = op[0]

        # ----------------------------------
        # EXPANSION / TILING TASKS
        # ----------------------------------
        if features["expansion"]:
            if name in [
                "tile_full",
                "crop_nonzero_bbox",
                "crop_to_bbox",
                "compress_vertical",
                "compress_horizontal",
                "isolate_objects",
                "extract_largest_object",
                "reflect_h",
                "reflect_v",
                "rotate90",
                "rotate180",
                "rotate270",
            ]:
                filtered.append(op)

        # ----------------------------------
        # SAME-SHAPE COLOR / OBJECT TASKS
        # ----------------------------------
        elif features["same_shape"] and features["color_change"]:
            if name in [
                "color_map",
                "conditional_color_map",
                "recolor_role",
                "translate_role",
                "extract_largest_object",
                "crop_nonzero_bbox",
            ]:
                filtered.append(op)

        # ----------------------------------
        # DEFAULT
        # ----------------------------------
        else:
            filtered.append(op)

    if not filtered:
        filtered = all_ops

    return filtered


# quick sanity check on first task
_first_task_id = list(solver_tasks.keys())[0]
_first_task = solver_tasks[_first_task_id]
_first_ops = build_task_ops(_first_task)

print("CELL 07 loaded — unified task-filtered operator registry active.")
print("First task id:", _first_task_id)
print("First task op count:", len(_first_ops))
print("First task first 20 ops:", _first_ops[:20])

CELL 07 loaded — unified task-filtered operator registry active.
First task id: 00576224
First task op count: 13
First task first 20 ops: [('rotate90', {}), ('rotate180', {}), ('rotate270', {}), ('reflect_h', {}), ('reflect_v', {}), ('tile_full', {'nx': 2, 'ny': 2}), ('tile_full', {'nx': 3, 'ny': 3}), ('crop_to_bbox', {}), ('extract_largest_object', {}), ('crop_nonzero_bbox', {}), ('isolate_objects', {}), ('compress_vertical', {}), ('compress_horizontal', {})]


In [8]:
# ============================================
# CELL 7b — FINALIZATION OPERATORS
# ============================================

import numpy as np

# ----------------------------------
# CROP NON-ZERO BOUNDING BOX
# ----------------------------------
def crop_nonzero_bbox(grid):
    g = np.array(grid)
    rows = np.any(g != 0, axis=1)
    cols = np.any(g != 0, axis=0)

    if not np.any(rows) or not np.any(cols):
        return g

    r = np.where(rows)[0]
    c = np.where(cols)[0]

    return g[r[0]:r[-1]+1, c[0]:c[-1]+1]


# ----------------------------------
# REMOVE BACKGROUND (SET TO 0 OUTSIDE OBJECTS)
# ----------------------------------
def isolate_objects(grid):
    g = np.array(grid)
    objs = extract_objects(g.tolist())

    out = np.zeros_like(g)

    for obj in objs:
        for r, c in obj["pixels"]:
            out[r, c] = g[r, c]

    return out


# ----------------------------------
# STACK OBJECTS (VERTICAL COMPRESSION)
# ----------------------------------
def compress_vertical(grid):
    g = np.array(grid)
    rows = [row for row in g if np.any(row != 0)]

    if not rows:
        return g

    return np.array(rows)


# ----------------------------------
# STACK OBJECTS (HORIZONTAL COMPRESSION)
# ----------------------------------
def compress_horizontal(grid):
    g = np.array(grid)
    cols = [g[:, i] for i in range(g.shape[1]) if np.any(g[:, i] != 0)]

    if not cols:
        return g

    return np.stack(cols, axis=1)


print("CELL 7b loaded — finalization operators ready.")

CELL 7b loaded — finalization operators ready.


In [9]:
# ============================================
# CELL 08 — PROGRAM EXECUTION (FULL)
# ============================================

def run_program(program, grid):

    current = to_np(grid)

    for op_name, params in program:

        try:
            # ----------------------------------
            # TRANSFORM OPS
            # ----------------------------------
            if op_name == "rotate90":
                current = rotate90(current)

            elif op_name == "rotate180":
                current = rotate180(current)

            elif op_name == "rotate270":
                current = rotate270(current)

            elif op_name == "reflect_h":
                current = reflect_h(current)

            elif op_name == "reflect_v":
                current = reflect_v(current)

            elif op_name == "tile_full":
                current = tile_full(current, params["nx"], params["ny"])

            # ----------------------------------
            # COLOR OPS
            # ----------------------------------
            elif op_name == "color_map":
                current = color_map(current, params["mapping"])

            elif op_name == "conditional_color_map":
                current = conditional_color_map(current, params["source"], params["target"])

            # ----------------------------------
            # STRUCTURE OPS
            # ----------------------------------
            elif op_name == "crop_to_bbox":
                current = crop_to_bbox(current)

            elif op_name == "extract_largest_object":
                current = extract_largest_object(current)

            # ----------------------------------
            # ROLE OPS
            # ----------------------------------
            elif op_name == "recolor_role":
                current = recolor_role(current, params["role"], params["color"])

            elif op_name == "translate_role":
                current = translate_role(current, params["role"], params["dx"], params["dy"])

            # ----------------------------------
            # 🔥 FINALIZATION OPS (NEW)
            # ----------------------------------
            elif op_name == "crop_nonzero_bbox":
                current = crop_nonzero_bbox(current)

            elif op_name == "isolate_objects":
                current = isolate_objects(current)

            elif op_name == "compress_vertical":
                current = compress_vertical(current)

            elif op_name == "compress_horizontal":
                current = compress_horizontal(current)

            # ----------------------------------
            # UNKNOWN OP SAFETY
            # ----------------------------------
            else:
                return None

        except Exception:
            return None

    return to_list(current)


print("CELL 08 loaded — execution engine ready.")

CELL 08 loaded — execution engine ready.


In [10]:
# ============================================
# CELL 09 — AOFE ERROR ANALYSIS + SCORING
# ============================================

def analyze_errors(pred, target):
    pred = to_list(pred)
    target = to_list(target)

    same = same_shape(pred, target)

    h = max(len(pred), len(target))
    w = max(len(pred[0]), len(target[0]))

    error_map = []
    error_count = 0

    for r in range(h):
        row = []
        for c in range(w):
            pv = pred[r][c] if r < len(pred) and c < len(pred[0]) else None
            tv = target[r][c] if r < len(target) and c < len(target[0]) else None
            mismatch = int(pv != tv)
            row.append(mismatch)
            error_count += mismatch
        error_map.append(row)

    total = len(target) * len(target[0]) if same else h * w
    error_density = error_count / max(1, total)
    pixel_accuracy = 1.0 - (error_count / max(1, total)) if same else 0.0

    pred_colors = set(v for row in pred for v in row)
    tgt_colors = set(v for row in target for v in row)
    color_score = len(pred_colors & tgt_colors) / max(1, len(pred_colors | tgt_colors))

    try:
        pred_objs = extract_objects(pred)
        tgt_objs = extract_objects(target)
        struct_count = 1.0 - abs(len(pred_objs) - len(tgt_objs)) / max(1, max(len(pred_objs), len(tgt_objs)))
    except Exception:
        struct_count = 0.0

    structural_score = 0.5 * float(same) + 0.5 * struct_count

    return {
        "error_map": error_map,
        "error_count": error_count,
        "error_density": max(0.0, min(1.0, error_density)),
        "pixel_accuracy": max(0.0, min(1.0, pixel_accuracy)),
        "color_score": max(0.0, min(1.0, color_score)),
        "structural_score": max(0.0, min(1.0, structural_score)),
    }

def calculate_aofe_score(pred, target):
    a = analyze_errors(pred, target)
    score = (
        0.60 * a["pixel_accuracy"] +
        0.25 * a["structural_score"] +
        0.15 * a["color_score"]
    )
    return max(0.0, min(1.0, score))

def eval_program_with_errors(program, task):
    analyses = []
    pair_scores = []

    for pair in task["train"]:
        pred = run_program(program, pair["input"])
        if pred is None:
            return None

        analysis = analyze_errors(pred, pair["output"])
        score = calculate_aofe_score(pred, pair["output"])

        analyses.append(analysis)
        pair_scores.append(score)

    return {
        "score": sum(pair_scores) / max(1, len(pair_scores)),
        "analyses": analyses,
        "pair_scores": pair_scores,
    }

print("CELL 09 loaded.")

CELL 09 loaded.


In [11]:
# ============================================
# CELL 10 — PRIORITY OPS
# ============================================

def get_priority_ops(analysis, ops):
    error_density = analysis["error_density"]
    pixel_accuracy = analysis["pixel_accuracy"]
    structural_score = analysis["structural_score"]

    transform_ops = []
    color_ops = []
    object_ops = []
    extract_ops = []
    other_ops = []

    for op in ops:
        name = op[0]

        if name in ["rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v", "tile_full"]:
            transform_ops.append(op)
        elif name in ["conditional_color_map", "color_map"]:
            color_ops.append(op)
        elif name in ["recolor_role", "translate_role"]:
            object_ops.append(op)
        elif name in ["crop_to_bbox", "extract_largest_object"]:
            extract_ops.append(op)
        else:
            other_ops.append(op)

    if error_density > 0.25:
        ordered = transform_ops + extract_ops + object_ops + color_ops + other_ops
    elif structural_score > 0.8 and pixel_accuracy < 1.0:
        ordered = color_ops + object_ops + transform_ops + extract_ops + other_ops
    else:
        ordered = object_ops + transform_ops + color_ops + extract_ops + other_ops

    return ordered

print("CELL 10 loaded.")

CELL 10 loaded.


In [12]:
# ============================================
# CELL 10b — ARCHETYPE-AWARE PRIORITY OPS
# ============================================

def detect_task_archetype(task):
    pair = task["train"][0]
    inp = pair["input"]
    out = pair["output"]

    in_h, in_w = len(inp), len(inp[0])
    out_h, out_w = len(out), len(out[0])

    # shape expansion / tiling clue
    if out_h > in_h or out_w > in_w:
        return "tiling_or_expansion"

    # color-change clue
    if (in_h == out_h) and (in_w == out_w):
        changed = 0
        same = 0
        for r in range(in_h):
            for c in range(in_w):
                if inp[r][c] == out[r][c]:
                    same += 1
                else:
                    changed += 1

        if changed > 0 and same > 0:
            return "localized_recolor"

    return "generic"


def get_priority_ops(analysis, ops, task=None):
    error_density = analysis["error_density"]
    pixel_accuracy = analysis["pixel_accuracy"]
    structural_score = analysis["structural_score"]

    transform_ops = []
    color_ops = []
    object_ops = []
    finalization_ops = []
    extraction_ops = []
    other_ops = []

    for op in ops:
        name = op[0]

        if name in ["rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v", "tile_full"]:
            transform_ops.append(op)
        elif name in ["conditional_color_map", "color_map"]:
            color_ops.append(op)
        elif name in ["recolor_role", "translate_role"]:
            object_ops.append(op)
        elif name in ["crop_nonzero_bbox", "compress_vertical", "compress_horizontal", "isolate_objects"]:
            finalization_ops.append(op)
        elif name in ["crop_to_bbox", "extract_largest_object"]:
            extraction_ops.append(op)
        else:
            other_ops.append(op)

    archetype = detect_task_archetype(task) if task is not None else "generic"

    # ----------------------------------
    # TILING / EXPANSION TASKS
    # ----------------------------------
    if archetype == "tiling_or_expansion":
        ordered = (
            transform_ops +
            finalization_ops +
            extraction_ops +
            object_ops +
            color_ops +
            other_ops
        )

    # ----------------------------------
    # STRUCTURE GOOD, PIXELS WRONG
    # ----------------------------------
    elif structural_score > 0.8 and pixel_accuracy < 1.0:
        ordered = (
            color_ops +
            object_ops +
            finalization_ops +
            transform_ops +
            extraction_ops +
            other_ops
        )

    # ----------------------------------
    # HIGH ERROR
    # ----------------------------------
    elif error_density > 0.25:
        ordered = (
            transform_ops +
            extraction_ops +
            object_ops +
            finalization_ops +
            color_ops +
            other_ops
        )

    # ----------------------------------
    # DEFAULT
    # ----------------------------------
    else:
        ordered = (
            object_ops +
            color_ops +
            finalization_ops +
            transform_ops +
            extraction_ops +
            other_ops
        )

    return ordered


print("CELL 10b loaded — archetype-aware priority ops active.")

CELL 10b loaded — archetype-aware priority ops active.


In [13]:
# ============================================
# CELL 11 — AOFE BEAM SEARCH
# ============================================

def normalize_program(program):
    return [op for op in program]

def make_hashable(obj):
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    if isinstance(obj, list):
        return tuple(make_hashable(x) for x in obj)
    return obj

def program_signature(program):
    return tuple((name, make_hashable(params)) for name, params in normalize_program(program))

def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores, best_analyses, best_refinement_steps

        p = normalize_program(program)
        sig = program_signature(p)

        if sig in seen:
            return None
        seen.add(sig)

        result = eval_program_with_errors(p, task)
        if result is None:
            return None

        raw_score = result["score"]
        analyses = result["analyses"]
        pair_scores = result["pair_scores"]

        mdl_penalty = 0.001 * len(p)
        final_score = raw_score - mdl_penalty

        if raw_score > best_score or (raw_score == best_score and (best_program is None or len(p) < len(best_program))):
            best_program = p
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return (final_score, -len(p), p, pair_scores, analyses, refinement_steps)

    initial_programs = [[]]
    for op in ops:
        initial_programs.append([op])

    ranked = []
    for prog in initial_programs:
        item = consider(prog, refinement_steps=0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    for step in range(depth):
        candidates = []

        for _, _, program, _, analyses, _ in ranked:
            if not analyses:
                continue

            worst = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst, ops)
            last_op = program[-1][0] if program else None

            for op in ordered_ops[:width]:
                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, refinement_steps=step + 1)
                if item is not None:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score >= 0.999:
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [round(a["error_density"], 4) for a in best_analyses],
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator",
        "ranked": ranked,
    }

print("CELL 11 loaded.")

CELL 11 loaded.


In [14]:
# ============================================
# CELL 11b — SOLUTION LOCK + EARLY STOP FIX
# ============================================

def is_perfect_solution(result):
    if result is None:
        return False

    # strict check
    return all(a["error_density"] == 0.0 for a in result["analyses"])


def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    PERFECT_SOLUTION = None

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores
        nonlocal best_analyses, best_refinement_steps, PERFECT_SOLUTION

        sig = program_signature(program)
        if sig in seen:
            return None
        seen.add(sig)

        result = eval_program_with_errors(program, task)
        if result is None:
            return None

        raw_score = result["score"]
        analyses = result["analyses"]
        pair_scores = result["pair_scores"]

        # 🚨 SOLUTION LOCK
        if is_perfect_solution(result):
            PERFECT_SOLUTION = {
                "program": program,
                "score": raw_score,
                "analyses": analyses,
                "pair_scores": pair_scores,
                "steps": refinement_steps
            }

        mdl_penalty = 0.001 * len(program)
        final_score = raw_score - mdl_penalty

        if raw_score > best_score or (
            raw_score == best_score and (
                best_program is None or len(program) < len(best_program)
            )
        ):
            best_program = program
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return (final_score, -len(program), program, pair_scores, analyses, refinement_steps)

    # initial programs
    initial_programs = [[]] + [[op] for op in ops]

    ranked = []
    for prog in initial_programs:
        item = consider(prog, 0)
        if item:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    for step in range(depth):

        # 🚨 EARLY STOP IF PERFECT FOUND
        if PERFECT_SOLUTION is not None:
            break

        candidates = []

        for _, _, program, _, analyses, _ in ranked:

            if not analyses:
                continue

            worst = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst, ops)

            last_op = program[-1][0] if program else None

            for op in ordered_ops[:width]:

                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, step + 1)

                if item:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

    # 🚨 RETURN PERFECT IF FOUND
    if PERFECT_SOLUTION is not None:
        return {
            "best_program": PERFECT_SOLUTION["program"],
            "best_score": 1.0,
            "pair_scores": PERFECT_SOLUTION["pair_scores"],
            "refinement_steps": PERFECT_SOLUTION["steps"],
            "error_density_progression": [0.0],
            "failure_mode": None,
            "ranked": ranked,
        }

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [round(a["error_density"], 4) for a in best_analyses],
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator",
        "ranked": ranked,
    }

print("CELL 11b loaded — solution lock active.")

CELL 11b loaded — solution lock active.


In [15]:
# ============================================
# VERIFY ACTIVE beam_search VERSION
# ============================================

import inspect

src = inspect.getsource(beam_search)

print("---- BEAM SEARCH SOURCE (FIRST 300 CHARS) ----")
print(src[:300])

print("\nContains solution lock?:", "PERFECT_SOLUTION" in src)
print("Contains early stop?:", "EARLY STOP" in src or "PERFECT_SOLUTION" in src)

---- BEAM SEARCH SOURCE (FIRST 300 CHARS) ----
def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    PERFECT_SOLUTION = None

    def consider(program, refinement_steps):
        n

Contains solution lock?: True
Contains early stop?: True


In [16]:
# ============================================
# CELL 11d — COMPOSITION SEED TEMPLATES
# ============================================

def seed_programs_from_task(task):
    ops = build_task_ops(task)
    features = detect_task_features(task)

    seeds = []

    # ----------------------------------
    # Always allow empty start
    # ----------------------------------
    seeds.append([])

    # ----------------------------------
    # Single-op seeds
    # ----------------------------------
    for op in ops:
        seeds.append([op])

    # ----------------------------------
    # Expansion / tiling task seeds
    # ----------------------------------
    if features["expansion"]:
        tile_ops = [op for op in ops if op[0] == "tile_full"]
        finalize_ops = [op for op in ops if op[0] in [
            "crop_nonzero_bbox",
            "crop_to_bbox",
            "compress_vertical",
            "compress_horizontal",
            "isolate_objects",
            "extract_largest_object",
        ]]

        geom_ops = [op for op in ops if op[0] in [
            "reflect_h", "reflect_v", "rotate90", "rotate180", "rotate270"
        ]]

        # tile -> finalize
        for t in tile_ops:
            for f in finalize_ops:
                seeds.append([t, f])

        # extract -> tile
        for f in finalize_ops:
            for t in tile_ops:
                seeds.append([f, t])

        # geom -> tile
        for g in geom_ops:
            for t in tile_ops:
                seeds.append([g, t])

        # geom -> tile -> finalize
        for g in geom_ops:
            for t in tile_ops:
                for f in finalize_ops[:3]:
                    seeds.append([g, t, f])

    # ----------------------------------
    # Same-shape color/object task seeds
    # ----------------------------------
    elif features["same_shape"] and features["color_change"]:
        color_ops = [op for op in ops if op[0] in [
            "color_map",
            "conditional_color_map",
        ]]
        role_ops = [op for op in ops if op[0] == "recolor_role"]
        move_ops = [op for op in ops if op[0] == "translate_role"]

        # color -> recolor
        for c in color_ops:
            for r in role_ops[:12]:
                seeds.append([c, r])

        # recolor -> move
        for r in role_ops[:12]:
            for m in move_ops[:8]:
                seeds.append([r, m])

        # color -> recolor -> move
        for c in color_ops[:10]:
            for r in role_ops[:8]:
                for m in move_ops[:4]:
                    seeds.append([c, r, m])

    # ----------------------------------
    # Deduplicate
    # ----------------------------------
    unique = []
    seen = set()

    for prog in seeds:
        sig = program_signature(prog)
        if sig not in seen:
            seen.add(sig)
            unique.append(prog)

    return unique


def is_perfect_solution(result):
    if result is None:
        return False
    return all(a["error_density"] == 0.0 for a in result["analyses"])


def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    perfect_solution = None

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores
        nonlocal best_analyses, best_refinement_steps, perfect_solution

        sig = program_signature(program)
        if sig in seen:
            return None
        seen.add(sig)

        result = eval_program_with_errors(program, task)
        if result is None:
            return None

        raw_score = result["score"]
        analyses = result["analyses"]
        pair_scores = result["pair_scores"]

        if is_perfect_solution(result):
            perfect_solution = {
                "program": program,
                "score": raw_score,
                "analyses": analyses,
                "pair_scores": pair_scores,
                "steps": refinement_steps,
            }

        mdl_penalty = 0.001 * len(program)
        final_score = raw_score - mdl_penalty

        if raw_score > best_score or (
            raw_score == best_score and (
                best_program is None or len(program) < len(best_program)
            )
        ):
            best_program = program
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return (final_score, -len(program), program, pair_scores, analyses, refinement_steps)

    # ----------------------------------
    # Seeded initial programs (NEW)
    # ----------------------------------
    initial_programs = seed_programs_from_task(task)

    ranked = []
    for prog in initial_programs:
        item = consider(prog, 0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    for step in range(depth):
        if perfect_solution is not None:
            break

        candidates = []

        for _, _, program, _, analyses, _ in ranked:
            if not analyses:
                continue

            worst = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst, ops, task=task)
            last_op = program[-1][0] if program else None

            for op in ordered_ops[:width]:
                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, step + 1)

                if item is not None:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

    if perfect_solution is not None:
        return {
            "best_program": perfect_solution["program"],
            "best_score": 1.0,
            "pair_scores": perfect_solution["pair_scores"],
            "refinement_steps": perfect_solution["steps"],
            "error_density_progression": [0.0],
            "failure_mode": None,
            "ranked": ranked,
        }

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [round(a["error_density"], 4) for a in best_analyses],
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator",
        "ranked": ranked,
    }


print("CELL 11d loaded — composition seed templates active.")

CELL 11d loaded — composition seed templates active.


In [17]:
# ============================================
# CELL 12 — RUN CONTROLLED TASK
# ============================================

def run_controlled_task(name, task, width=12, depth=4):
    print("=" * 70)
    print(f"TEST: {name}")
    print("=" * 70)

    result = beam_search(task, width=width, depth=depth)

    print("\nBest program:")
    print(result["best_program"])
    print(f"Score: {result['best_score']:.4f}")
    print(f"Refinement steps: {result['refinement_steps']}")
    print(f"Error density progression: {result['error_density_progression']}")
    print(f"Failure mode: {result['failure_mode']}")

    return result

print("CELL 12 loaded.")

CELL 12 loaded.


In [18]:
# ============================================
# CELL 13 — PERFORMANCE SUMMARY METRICS
# ============================================

def summarize_results(results):
    scores = []
    solves = 0

    for task_id, res in results:
        score = res["best_score"]
        scores.append(score)

        if score >= 0.999:
            solves += 1

    avg_score = sum(scores) / len(scores) if scores else 0
    solve_rate = solves / len(scores) if scores else 0

    print("\n" + "=" * 70)
    print("PERFORMANCE SUMMARY")
    print("=" * 70)
    print(f"Tasks evaluated: {len(scores)}")
    print(f"Average score: {avg_score:.4f}")
    print(f"Solve rate: {solve_rate:.2%}")
    print("=" * 70)

    return {
        "avg_score": avg_score,
        "solve_rate": solve_rate,
        "total_tasks": len(scores)
    }

print("CELL 13 loaded.")

CELL 13 loaded.


In [19]:
# ============================================
# CELL 14 — CONNECT GOOGLE DRIVE
# ============================================

from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [20]:
# ============================================
# CELL 15 — LOGGING SYSTEM SETUP
# ============================================

import os
import json
from datetime import datetime

LOG_DIR = "/content/drive/MyDrive/AOFE_LOGS"
LOG_FILE = os.path.join(LOG_DIR, "aofe_progress_log.json")

# create directory if needed
os.makedirs(LOG_DIR, exist_ok=True)

# initialize file if not exists
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        json.dump([], f)

print("Log system ready.")
print("Log file:", LOG_FILE)

Log system ready.
Log file: /content/drive/MyDrive/AOFE_LOGS/aofe_progress_log.json


In [21]:
# ============================================
# CELL 16 — LOG RESULTS FUNCTION
# ============================================

def log_run(results, config):

    with open(LOG_FILE, "r") as f:
        logs = json.load(f)

    entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "config": config,
        "summary": [],
    }

    for task_id, res in results:
        entry["summary"].append({
            "task_id": task_id,
            "score": res["best_score"],
            "steps": res["refinement_steps"],
            "failure": res["failure_mode"],
            "program_length": len(res["best_program"]) if res["best_program"] else 0
        })

    logs.append(entry)

    with open(LOG_FILE, "w") as f:
        json.dump(logs, f, indent=2)

    print("✅ Run logged successfully.")

In [22]:
# ============================================
# CELL 17 — BENCHMARK (DEEP SEARCH MODE)
# ============================================

results = []

for i, (task_id, task) in enumerate(solver_tasks.items()):
    if i >= 5:
        break

    print("\n" + "=" * 70)
    print(f"TASK {i} | ID: {task_id}")
    print("=" * 70)

    result = run_controlled_task(
        f"ARC Task {task_id}",
        task,
        width=30,     # 🔥 increased
        depth=6       # 🔥 increased
    )

    results.append((task_id, result))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

for task_id, res in results:
    print(f"\nTask: {task_id}")
    print(f"  Score: {res['best_score']:.4f}")
    print(f"  Steps: {res['refinement_steps']}")
    print(f"  Failure: {res['failure_mode']}")
    print(f"  Program: {res['best_program']}")

# ----------------------------------
# METRICS
# ----------------------------------
summary_stats = summarize_results(results)

# ----------------------------------
# LOGGING
# ----------------------------------
config = {
    "width": 30,
    "depth": 6,
    "mode": "deep_search_unlock",
    "notes": "enable finalization operators"
}

log_run(results, config)


TASK 0 | ID: 00576224
TEST: ARC Task 00576224

Best program:
[('tile_full', {'nx': 3, 'ny': 3})]
Score: 0.7896
Refinement steps: 0
Error density progression: [0.3333, 0.3333]
Failure mode: near_miss_or_missing_operator

TASK 1 | ID: 007bbfb7
TEST: ARC Task 007bbfb7

Best program:
[('extract_largest_object', {}), ('tile_full', {'nx': 3, 'ny': 3})]
Score: 0.8515
Refinement steps: 0
Error density progression: [0.2716, 0.1481, 0.1852, 0.2716, 0.1728]
Failure mode: near_miss_or_missing_operator

TASK 2 | ID: 009d5c81
TEST: ARC Task 009d5c81

Best program:
[('conditional_color_map', {'source': 1, 'target': 0}), ('recolor_role', {'role': 'largest', 'color': 2})]
Score: 0.8977
Refinement steps: 0
Error density progression: [0.1582, 0.0714, 0.0, 0.0816, 0.0]
Failure mode: near_miss_or_missing_operator

TASK 3 | ID: 00d62c1b
TEST: ARC Task 00d62c1b

Best program:
[('conditional_color_map', {'source': 0, 'target': 4}), ('recolor_role', {'role': 'largest', 'color': 0})]
Score: 0.9618
Refinement s